## Imports and Setup

In [ ]:
from graphs import generate_grid_graph, grid_graph_max_cut, grid_renderer, state_history_renderer, render_state, render_state_prog_gif
from ising import Ising
from cim import CIM
from solvers import SG3
from transfers import TraditionalDOPO, Clipped
from graphs import read_graph_from_rudy, eval_max_cut
from schedules import schedule_linear
import torch
import numpy as np
import matplotlib.pyplot as plt

%pip install matplotlib-label-lines
from labellines import labelLines

%matplotlib inline

In [ ]:
size = 20
J = torch.zeros((size, size))
max_cut = 0
print(f"{max_cut=}")
pump_schedule= lambda t: 0.01 * (t - 100) + 1
noise_magnitude = 0.001; steps=30000; step_size=0.01

## Transfer Function Descriptions

In [ ]:
points = 131

points_J = torch.zeros((points, points))
foo = Ising(points_J)
foo.state += torch.from_numpy(np.linspace(-1.2, 1.2, points, endpoint=True))

fig, ax1 = plt.subplots()

dopo_transfer = TraditionalDOPO(pump_schedule=pump_schedule)


color = 'tab:blue'
ax1.set_xlabel('CIM Spin Magnitude (a.u.)')
ax1.set_xlim(-1.15, 1.15)
ax1.set_ylabel('DOPO Gradient (a.u.)', color=color)
ax1.set_ylim(-1.15, 1.15)
for time, linestyle, label in ((100, 'solid', 'p=1.0'), (200, 'dashed', 'p=2.0'), (300, 'dotted', 'p=3.0')):
    dopo_grad = dopo_transfer.grad(foo, time=time)
    ax1.plot(foo.state, dopo_grad, color=color, linestyle=linestyle, label=label)
ax1.tick_params(axis='y', labelcolor=color)

lines = plt.gca().get_lines()
labelLines(lines, align=True, xvals=(-0.6, -0.7, -0.6))

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

time = 200
clipped_transfer = Clipped(pump_schedule=pump_schedule)
clipped_grad = clipped_transfer.grad(foo, time=time)

color = 'tab:red'
ax2.set_ylabel('Clipped Gradient (a.u.)', color=color)
ax2.set_ylim(-1.15, 1.15)
clipped_line = ax2.plot(foo.state, clipped_grad, color=color, label="p=2.0")
labelLines(clipped_line, align=True, xvals=(0.6))
ax2.tick_params(axis='y', labelcolor=color)

ax1.legend(lines + clipped_line, ["DOPO p=1.0", "DOPO p=2.0", "DOPO p=3.0", "Clipped p=2.0"], loc=9)
fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.savefig('ims/DOPO_vs_Clipped_Transfer_Function.png', dpi=900)


## DOPO Transfer Function

In [ ]:
foo = CIM(-0.1 * J, pump_schedule=pump_schedule)
foo.solve(noise_magnitude = 0.001, steps=30000, step_size=0.01)
print(foo.model.state)
print("DOPO:", eval_max_cut(foo.model.state.tolist(), J))

In [ ]:
t = np.linspace(0, 0.01*30000, 30000)

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax1.set_xlabel('time (a.u.)')
ax1.set_ylabel('CIM Spins (a.u.)', color=color)  # we already handled the x-label with ax1
ax1.plot(t, foo.model.result.state_history, color=color)
ax1.tick_params(axis='y', labelcolor=color)

color = 'tab:red'
ax2.set_ylabel('Pump Rate (a.u.)', color=color)
ax2.plot(t, pump_schedule(t), color=color)
ax2.tick_params(axis='y', labelcolor=color)

ax2.axvline(100, color='k', linestyle='--')
ax2.axhline(1, color='k', linestyle='--')

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.savefig('ims/DOPO_CIM_spins_pump_rate.png', dpi=900)

## Clipped Transfer

### Within a CIM

In [ ]:
foo = CIM(-0.1 * J, pump_schedule=pump_schedule, transfer=Clipped)
foo.solve(noise_magnitude = 0.001, steps=30000, step_size=0.01)
print(foo.model.state)
print("DOPO:", eval_max_cut(foo.model.state.tolist(), J))

In [ ]:
t = np.linspace(0, 0.01*30000, 30000)

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax1.set_xlabel('time (a.u.)')
ax1.set_ylabel('CIM Spins (a.u.)', color=color)  # we already handled the x-label with ax1
ax1.plot(t, foo.model.result.state_history, color=color)
ax1.tick_params(axis='y', labelcolor=color)

color = 'tab:red'
ax2.set_ylabel('Pump Rate (a.u.)', color=color)
ax2.plot(t, pump_schedule(t), color=color)
ax2.tick_params(axis='y', labelcolor=color)

ax2.axvline(100, color='k', linestyle='--')
ax2.axhline(1, color='k', linestyle='--')

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.savefig('ims/Clipped_CIM_spins_pump_rate.png', dpi=900)